# Google Colab GPU — FIDES full-session showcase

Chọn **Runtime → Change runtime type → T4 GPU**, sau đó chạy lần lượt từng cell. Notebook chỉ cài wheel đã được GitLab CI build; Colab không compile OpenFHE hoặc FIDESlib.

## 1. Kiểm tra GPU

In [ ]:
import subprocess
import sys

print("Python:", sys.version)
subprocess.run(["nvidia-smi"], check=True)

if sys.version_info[:2] != (3, 12):
    raise RuntimeError("he-sdk-fides hiện hỗ trợ đúng Python 3.12")

## 2. Cài GPU SDK

Core package lấy từ public PyPI; FIDES native wheel lấy từ GitLab Package Registry. Không cài extra `[cpu]` hoặc package `openfhe` trong runtime này.

In [ ]:
%pip install --upgrade pip
%pip install --extra-index-url "https://gitlab.com/api/v4/projects/84844502/packages/pypi/simple" "he_looming_sdk[gpu]==0.6.5"

## 3. Tạo GPU session

In [ ]:
from he_sdk import HESession, __version__
import he_sdk_fides

session = HESession.create(device="gpu")

left_values = [1.0, 2.0, 3.0, 4.0]
right_values = [10.0, 20.0, 30.0, 40.0]

print("he_sdk version:", __version__)
print("he_sdk_fides version:", he_sdk_fides.__version__)
print("selected backend:", session.capabilities.backend)
print("capabilities:", session.capabilities)

## 4. Encrypt và decrypt trên FIDES GPU

In [ ]:
left_ct = session.encrypt(left_values)
right_ct = session.encrypt(right_values)

print("left input:", left_values)
print("encrypted left:", left_ct)
print("decrypted left:", session.decrypt(left_ct))

## 5. Các phép toán GPU

In [ ]:
add_ct = session.add(left_ct, right_ct)
subtract_ct = session.subtract(left_ct, right_ct)
multiply_ct = session.multiply(left_ct, right_ct)
square_ct = session.square(left_ct)
sum_ct = session.sum(left_ct)
mean_ct = session.mean(left_ct)
variance_ct = session.variance(left_ct)

print("add expected:", [11.0, 22.0, 33.0, 44.0])
print("add decrypted:", session.decrypt(add_ct))
print("subtract expected:", [-9.0, -18.0, -27.0, -36.0])
print("subtract decrypted:", session.decrypt(subtract_ct))
print("multiply expected:", [10.0, 40.0, 90.0, 160.0])
print("multiply decrypted:", session.decrypt(multiply_ct))
print("square expected:", [1.0, 4.0, 9.0, 16.0])
print("square decrypted:", session.decrypt(square_ct))
print("sum expected:", 10.0)
print("sum decrypted:", session.decrypt(sum_ct))
print("mean expected:", 2.5)
print("mean decrypted:", session.decrypt(mean_ct))
print("variance expected:", 1.25)
print("variance decrypted:", session.decrypt(variance_ct))

## 6. Đóng session

Local FIDES backend hiện chưa hỗ trợ workspace serialization và recipient/PRE.

In [ ]:
session.close()
print("GPU session closed")